# Self-Attention & Scaled Dot-Product Attention — Study Notes

## 1. The Core Idea

Self-attention lets every position in a sequence look at every other position and decide, dynamically, how much "attention" to pay to each one. Instead of a fixed window (like a CNN) or a sequential bottleneck (like an RNN), each token builds its output as a weighted mixture of *all* tokens in the sequence — the weights being learned, content-dependent, and computed in parallel.

---

## 2. Q, K, V — Query, Key, Value

For every input token embedding `x_i`, three vectors are produced using three learned weight matrices:

- **Query (Q)** = `X · W_Q` — "what am I looking for?"
- **Key (K)** = `X · W_K` — "what do I contain / how do I advertise myself?"
- **Value (V)** = `X · W_V` — "what information do I actually give out if selected?"

Intuition: think of it like a soft dictionary lookup. Each token emits a **query** (its search request), every token also emits a **key** (a label describing it), and the similarity between a query and each key tells you how much of that token's **value** to pull in.

- `X` has shape `(seq_len, d_model)`
- `W_Q`, `W_K` have shape `(d_model, d_k)`
- `W_V` has shape `(d_model, d_v)` (often `d_k = d_v`)
- So `Q`, `K` → `(seq_len, d_k)` and `V` → `(seq_len, d_v)`

---

## 3. Score Computation: `QK^T / √d_k`

**Step 1 — raw similarity:**
`scores = Q · K^T` → shape `(seq_len, seq_len)`

This is just a dot product between every query and every key. A dot product is large when two vectors point in similar directions — so this measures how relevant each key is to each query.

**Step 2 — scale by `√d_k`:**
`scaled_scores = (Q · K^T) / √d_k`

### Why scale? (Variance analysis)

Assume the components of `Q` and `K` are independent random variables with mean 0 and variance 1 (a reasonable assumption right after layer norm / initialization). For a single dot product:

`q · k = Σ_{i=1}^{d_k} q_i * k_i`

Each term `q_i * k_i` has:
- Mean = `E[q_i] * E[k_i] = 0`
- Variance = `E[q_i² k_i²] = E[q_i²] * E[k_i²] = 1 * 1 = 1` (independence)

Since it's a sum of `d_k` independent, zero-mean, unit-variance terms:

`Var(q · k) = Σ Var(q_i k_i) = d_k`

So the standard deviation of the raw dot product grows as `√d_k`. For a typical transformer, `d_k = 64`, so the raw scores have a standard deviation of `√64 = 8` — the values can get quite large.

**Why this is a problem:** these scores are fed into a softmax. Softmax is extremely sensitive to the scale of its inputs — if the scores have large magnitude, softmax saturates: the largest score gets a probability close to 1, and the rest collapse to nearly 0. This produces a near one-hot distribution, and because softmax's gradient is `p_i(1 - p_i)` (roughly), when `p_i → 0` or `p_i → 1` the gradient vanishes → the model becomes very hard to train.

**The fix:** dividing by `√d_k` normalizes the variance back to ~1 regardless of `d_k`, keeping softmax in a well-conditioned regime with healthy gradients. This is literally where the term "scaled" in Scaled Dot-Product Attention comes from — it's introduced explicitly in the *Attention Is All You Need* paper as a correction for exactly this variance blow-up.

---

## 4. Softmax

`attention_weights = softmax(scaled_scores, dim=-1)`

Applied row-wise (per query), softmax converts the scaled scores into a probability distribution over all keys — non-negative, summing to 1 per row:

`softmax(z_i) = e^{z_i} / Σ_j e^{z_j}`

Each row `i` now tells you: "for query token `i`, how much attention (0 to 1, summing to 1) should be paid to every token `j` in the sequence."

---

## 5. Output: Weighted Sum of V

`Output = attention_weights · V`

This is the final step: use the probability distribution as mixing weights over the **value** vectors.

`shape: (seq_len, seq_len) · (seq_len, d_v) → (seq_len, d_v)`

So each output row is a weighted combination of *all* value vectors, where the weights came from how well that token's query matched every other token's key. This is why self-attention is often summarized as one equation:

```
Attention(Q, K, V) = softmax( QK^T / √d_k ) · V
```

---

## 6. Masking

Masking modifies the scores **before** softmax by setting positions that must be ignored to `-∞` (in practice, a very large negative number like `-1e9`), so that after softmax those positions become ~0 probability.

### a) Padding mask
Sequences in a batch are padded to equal length. The model must not attend to `<PAD>` tokens (they carry no real information). A padding mask sets scores for all padding positions to `-∞` along the key dimension, so no query ever attends to a pad token.

### b) Causal (look-ahead) mask
Used in decoders / autoregressive models (e.g. GPT-style, or the decoder self-attention in the Transformer) to prevent a position from attending to future positions — otherwise the model could "cheat" by seeing the answer it's supposed to predict. This is implemented as an upper-triangular mask: position `i` can only attend to positions `≤ i`.

```
mask = [[0, -inf, -inf, -inf],
        [0,   0,  -inf, -inf],
        [0,   0,    0,  -inf],
        [0,   0,    0,    0]]

scaled_scores = scaled_scores + mask
```

Both masks are simply added to `scaled_scores` before the softmax step.

---

## 7. Putting It All Together (pseudocode)

```python
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1)      # QK^T
    scores = scores / sqrt(d_k)           # scale
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = softmax(scores, dim=-1)     # attention weights
    output = weights @ V                  # weighted sum of V
    return output, weights
```

Multi-head attention just runs this in parallel across `h` heads with smaller `d_k = d_model / h`, then concatenates and projects the results — letting different heads specialize in different types of relationships.

---

## 8. Reading List — Chronological Order

These trace the actual historical path: RNN seq2seq → attention as a fix for its bottleneck → attention as the *entire* architecture → modern survey of everywhere it's used since.

| # | Year | Paper | Authors | Link |
|---|------|-------|---------|------|
| 1 | 2014 | Sequence to Sequence Learning with Neural Networks | Sutskever, Vinyals, Le | https://arxiv.org/abs/1409.3215 |
| 2 | 2014 | Neural Machine Translation by Jointly Learning to Align and Translate | Bahdanau, Cho, Bengio | https://arxiv.org/abs/1409.0473 |
| 3 | 2017 | Attention Is All You Need | Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Gomez, Kaiser, Polosukhin | https://arxiv.org/abs/1706.03762 |
| 4 | 2023 | A Comprehensive Survey on Applications of Transformers for Deep Learning Tasks | Islam, Elmekki, Elsebai, Bentahar, Drawel, Rjoub, Pedrycz | https://arxiv.org/abs/2306.07303 |

**How they connect:**
1. **Sutskever et al. (2014)** introduced the encoder–decoder LSTM seq2seq framework — but it compresses the *entire* input sentence into a single fixed-length vector, which is a bottleneck for long sequences.
2. **Bahdanau et al. (2014)** fixed that bottleneck by introducing an *attention* mechanism — the decoder learns to "softly search" and align to relevant parts of the source sentence at every decoding step, instead of relying on one fixed vector. This is the direct conceptual ancestor of Q/K/V attention.
3. **Vaswani et al. (2017)** removed the recurrence entirely — showing that self-attention alone (no RNN/CNN) is sufficient, and formalized the exact Scaled Dot-Product Attention and Multi-Head Attention mechanisms covered above. This is the Transformer paper.
4. **Islam et al. (2023)** surveys everything transformers have since been applied to — NLP, computer vision, audio/speech, healthcare, IoT, etc. — good for seeing the breadth of downstream impact after reading the foundational three.